# Colab tester: score fine-tuned adapters through the repo's CLI

The runner's sibling, without training. It scores adapters already on the Hub, several per
session, and turns them into the handoff's validation scoreboard.

- **Score:** every `(config, run)` in `RUNS` goes through `python -m src.cli.inspect_llm`, the
  same pass the runner uses. Each CSV is copied to Drive as soon as it's written.
- **Reuse:** with `REUSE_DRIVE_CSVS`, a split whose CSV is already on Drive
  (`outputs/llm/eval/<run>/inspection_<split>.csv`) is copied back instead of rescored. A
  scoreboard of runs that are all scored already needs no GPU time.
- **Scoreboard:** the numbers come after Platt recalibration, as
  `inspect_llm --calibrate-with validation` computes them. Validation is cross-fitted, and
  test is fitted on the same run's validation. Bars: **0.659** shows training does anything
  (base-8b-5shot), and **0.628** beats test_run2.
- **Test is scored once** (E15). Putting `"test"` in `SPLITS` also needs `SCORE_TEST = True`.
- **Playground:** loads one adapter in this notebook and shows its answer for a few games.

In [ ]:
# ---- What to score ----
# (config under configs/train_llm/, run name = Hub repo / eval folder[, extra inspect_llm args])
RUNS = [
    ("qlora/llama32_1b/v2", "e4-llama32-1b"),
    # ("qlora/llama32_3b/v2", "e5-llama32-3b"),
    # ("qlora/llama31_8b/v2", "base-8b-5shot", "--base-only --few-shot 5 --batch-size 8"),
]
SPLITS = ["validation"]                 # add "test" only for the final model (E15)
SCORE_TEST = False                      # the second key needed to score "test"
SIZE = 5000                             # rows per split; 5000 covers all of validation and test
REUSE_DRIVE_CSVS = True                 # False = rescore even when Drive has the CSV
RELEASE_RUNTIME_WHEN_DONE = True        # release the GPU after the last cell (or on a failed score)

# One adapter, a few games, run in this notebook. None skips it.
PLAYGROUND = None
# PLAYGROUND = {
#     "config": "qlora/llama32_1b/v2",
#     "run": "e4-llama32-1b",
#     "split": "validation",
#     "game_ids": [],                     # empty = the first n games of the split
#     "n": 3,
#     "base_only": False,                 # True scores the untuned base model
#     "show_prompt": False,
# }

In [ ]:
# ---- Guard: test is scored once (E15), and only on purpose ----
# Runs before the slow install, so a mistake costs nothing.
wants_test = "test" in SPLITS or (PLAYGROUND or {}).get("split") == "test"
if wants_test and not SCORE_TEST:
    raise ValueError("'test' is scored once, for the final model (E15). Set SCORE_TEST = True to score it.")
# Test is recalibrated with a Platt fit on the same run's validation, so validation comes first.
if "test" in SPLITS:
    SPLITS = list(dict.fromkeys(["validation", *SPLITS]))
RUNS = [(run[0], run[1], run[2] if len(run) > 2 else "") for run in RUNS]
for config, run_name, args in RUNS:
    print(f"{run_name:<24} {config:<28} {args}")
print("splits:", SPLITS)

In [ ]:
# ---- Setup: Drive -> local disk, install ----
# Fail before the slow install if Colab gave us a CPU runtime
# (fix: Runtime -> Change runtime type -> L4 GPU, then run all cells again).
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 GPU"
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
DRIVE_REPO = "/content/drive/MyDrive/nba_bets"
DRIVE_EVAL = f"{DRIVE_REPO}/outputs/llm/eval"
# .gdoc files are Google Docs shortcuts, not files: rsync cannot read them.
!rsync -a --delete --exclude outputs --exclude .venv --exclude mlruns --exclude '*.gdoc' {DRIVE_REPO}/ /content/nba_bets/
%cd /content/nba_bets
!pip install -q .[gpu]
# Colab preinstalls torchao 0.10. peft >= 0.21 raises on any torchao < 0.16 when
# it wraps an unquantized model (bf16 runs). Nothing here uses torchao.
!pip uninstall -y -q torchao

In [ ]:
# ---- Secrets -> env (`!` subprocesses cannot read Colab Secrets themselves) ----
# Inspection reads the gated base model and the adapter from the Hub; it logs nothing to W&B.
import os
from google.colab import userdata
os.environ["HF_llm_training_token"] = userdata.get("HF_llm_training_token")

In [ ]:
# ---- Helpers: run a CLI step, release the runtime ----
# Colab's `!` does not report a failed command back to Python, so a crashed
# step would fall through to the next one. run() streams the output live
# (progress bars included) and raises on a non-zero exit, stopping the cell.
# Pressing Stop interrupts only this notebook's Python, not the child process,
# so run() stops the child itself - otherwise inference keeps going unseen.
import os, signal, subprocess, sys

def run(cmd):
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True
    )
    try:
        for chunk in iter(lambda: proc.stdout.read1(4096), b""):
            sys.stdout.write(chunk.decode(errors="replace"))
            sys.stdout.flush()
    except KeyboardInterrupt:
        os.killpg(proc.pid, signal.SIGINT)   # like Ctrl+C in a terminal
        try:
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        print(f"\nStopped: {cmd}", flush=True)
        raise
    if proc.wait() != 0:
        raise RuntimeError(f"Failed (exit code {proc.returncode}): {cmd}")


def release():
    drive.flush_and_unmount()   # without this, recent Drive writes can be lost
    from google.colab import runtime
    runtime.unassign()          # disconnects and deletes the runtime; billing stops

In [ ]:
# ---- Score: inspect_llm per run and split (GPU), each CSV saved to Drive at once ----
import shutil
from pathlib import Path

try:
    for config, run_name, args in RUNS:
        cfg = f"configs/train_llm/{config}.yaml"
        local_dir = Path("outputs/llm/eval") / run_name
        drive_dir = Path(DRIVE_EVAL) / run_name
        for split in SPLITS:
            csv = f"inspection_{split}.csv"
            if REUSE_DRIVE_CSVS and (drive_dir / csv).exists():
                local_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy(drive_dir / csv, local_dir / csv)
                print(f"Reused {drive_dir / csv}")
                continue
            run(f"python -m src.cli.inspect_llm --config {cfg} --run-name {run_name} --split {split} --size {SIZE} {args}")
            # Copied per split, so a later failure doesn't lose the runs already scored.
            drive_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy(local_dir / csv, drive_dir / csv)
except Exception:
    # A failed Run all must not leave the GPU idle. Pressing Stop (KeyboardInterrupt)
    # keeps the runtime: you're at the notebook.
    if RELEASE_RUNTIME_WHEN_DONE:
        release()
    raise

In [ ]:
# ---- Scoreboard (CPU): raw and Platt-recalibrated, as inspect_llm --calibrate-with ----
import datetime
import pandas as pd
from IPython.display import display
from src.ml.config.loader import load_llm_training_config
from src.ml.llm.diagnostics import read_inspection_csv, recalibrate, summarize

FLOOR, BAR = 0.659, 0.628   # validation calibrated log-loss: base-8b-5shot, test_run2


def summaries(config, run_name):
    """{split: (raw summary, recalibrated summary)} from the run's saved CSVs."""
    seed = load_llm_training_config(f"configs/train_llm/{config}.yaml").evaluation.seed
    frames = {s: read_inspection_csv(f"outputs/llm/eval/{run_name}/inspection_{s}.csv") for s in SPLITS}
    out = {}
    for split, frame in frames.items():
        # Test uses the fit on validation; any other split is cross-fitted in two halves.
        fit_on = frames["validation"] if split == "test" else None
        out[split] = (summarize(frame), summarize(recalibrate(frame, fit_on, seed=seed)))
    return out


SUMMARIES = {run_name: summaries(config, run_name) for config, run_name, _ in RUNS}
rows = []
for run_name, by_split in SUMMARIES.items():
    for split, (raw, cal) in by_split.items():
        rows.append({
            "run": run_name,
            "split": split,
            "n": raw["n"],
            "accuracy": cal["accuracy_p_home"],
            "log_loss": cal["log_loss"],
            "log_loss_raw": raw["log_loss"],
            "brier": cal["brier"],
            "auc": raw.get("auc_p_home"),            # Platt keeps the ranking
            "picked_home": cal["share_predicted_home"],
            "picked_home_raw": raw["share_predicted_home"],
            "home_rate": raw["truth_home_win_rate"],
            "sign_mass": raw["mean_p_sign_mass"],
            "rmse_expected": raw.get("rmse_points_expected"),
            "rmse_greedy": raw.get("rmse_points_greedy"),
            "rmse_mean": raw.get("rmse_points_predict_mean"),
        })
SCOREBOARD = pd.DataFrame(rows).sort_values(["split", "log_loss"]).reset_index(drop=True)
display(SCOREBOARD.style.format(precision=3, na_rep="-"))
print(f"Validation log-loss bars: < {FLOOR} shows training does anything, < {BAR} beats test_run2.")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
SCOREBOARD.to_csv(f"{DRIVE_EVAL}/scoreboard_{stamp}.csv", index=False)
print(f"Saved {DRIVE_EVAL}/scoreboard_{stamp}.csv")

In [ ]:
# ---- Calibration plot (CPU): predicted vs actual home-win rate per bin ----
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(SPLITS), figsize=(6.5 * len(SPLITS), 6), squeeze=False)
for ax, split in zip(axes[0], SPLITS):
    ax.plot([0, 1], [0, 1], color="grey", lw=1, ls=":")
    for i, (run_name, by_split) in enumerate(SUMMARIES.items()):
        raw, cal = by_split[split]
        for summary, style, label in ((cal, "-", run_name), (raw, "--", None)):
            table = pd.DataFrame(summary["calibration"])
            alpha = 1.0 if style == "-" else 0.45
            ax.plot(table.mean_p, table.win_rate, style, color=f"C{i}", alpha=alpha, label=label)
            # Marker area follows the bin's share of games: sparse bins are noisy.
            ax.scatter(table.mean_p, table.win_rate, s=20 + 400 * table.n / table.n.sum(), color=f"C{i}", alpha=alpha)
    ax.set(
        title=f"{split}: solid = Platt, dashed = raw",
        xlabel="mean predicted P(home win)",
        ylabel="actual home-win rate",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Playground (GPU, in this notebook): one adapter, a few games ----
if PLAYGROUND:
    import gc
    import torch
    from src.ml.llm.dataset import load_hf_split
    from src.ml.llm.diagnostics import (
        _home_won,
        apply_platt,
        fit_platt,
        magnitude_token_ids,
        predict_with_first_token_probs,
    )
    from src.ml.llm.evaluate import _load_adapter_model
    from src.ml.llm.finetune import build_hub_model_id

    pg = PLAYGROUND
    config = load_llm_training_config(f"configs/train_llm/{pg['config']}.yaml")
    field = config.data.prompt_field
    data = load_hf_split(config, pg["split"])
    if pg.get("game_ids"):
        wanted = {str(g) for g in pg["game_ids"]}
        data = data.filter(lambda row: str(row["game_id"]) in wanted)
    else:
        data = data.select(range(min(pg.get("n", 3), len(data))))
    games = [data[i] for i in range(len(data))]

    hub_id = None if pg.get("base_only") else build_hub_model_id(config, pg["run"])
    tokenizer, model = _load_adapter_model(config, hub_id)
    tokenizer.padding_side = "left"   # every prompt must end at the same column
    magnitude_ids = magnitude_token_ids(tokenizer)
    batch = config.evaluation.batch_size
    results = []
    for start in range(0, len(games), batch):
        results.extend(
            predict_with_first_token_probs(
                model,
                tokenizer,
                games[start : start + batch],
                prompt_field=field,
                max_new_tokens=config.evaluation.max_new_tokens,
                magnitude_ids=magnitude_ids,
            )
        )

    # Platt fitted on this run's whole validation CSV, when it was scored above.
    # Validation games shown here are then part of their own fit: a peek, not a score.
    val_csv = Path(f"outputs/llm/eval/{pg['run']}/inspection_validation.csv")
    platt = None
    if val_csv.exists():
        val = read_inspection_csv(val_csv)
        platt = fit_platt(val["p_home"], _home_won(val))

    shown = pd.DataFrame(results)
    shown["p_home_platt"] = apply_platt(shown["p_home"], *platt) if platt else float("nan")
    shown["expected_margin_pts"] = shown["expected_margin"] * shown["margin_scale"]
    if pg.get("show_prompt"):
        for game in games:
            print(f"--- game {game['game_id']} ---\n{game[field]}{game['completion']}\n")
    display(
        shown[["game_id", "truth", "margin", "p_home", "p_home_platt", "expected_margin_pts", "raw_output", "p_sign_mass", "top_first_tokens"]]
        .style.format(precision=3, na_rep="-")
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("PLAYGROUND is None: skipped.")

In [ ]:
# ---- End of session ----
# Drive uploads in the background; flushing makes sure the last copies reach Drive.
if RELEASE_RUNTIME_WHEN_DONE:
    release()
else:
    drive.flush_and_unmount()   # then disconnect the runtime yourself